In [1]:
import os
import pandas as pd
import numpy as np

# Softmax fusion temperature(tau) 그리드 서치 결과를 비교한다.
# 참고: run_cue_tau_1.sh / run_cue_tau_2.sh
# CUE 기본값(temperature=1.0)은 그리드의 TAU=1.0과 수치가 완전히 동일하므로
# (동일 시드로 결정론적 학습되어 사실상 같은 실행) 별도 행 없이 그리드 결과만 다룬다.
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# run_cue_tau_1.sh / _2.sh의 TEMPERATURES와 동일하게 맞춘다.
TEMPERATURES = [0.25, 0.5, 1.0, 2.0, 4.0]

In [2]:
def load_metric(dataset, target, temp):
    """cmfe의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다."""
    model_dir = os.path.join(cue_base, dataset, target, 'v1', f'TAU{temp}')
    metrics_path = os.path.join(model_dir, 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for temp in TEMPERATURES:
            vals = load_metric(d_name, target, temp)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'TAU': temp, **vals})

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다 (기대값: 2 x 4 x 5 = 40).")

✅ 데이터 로드 완료: 40개의 항목이 수집되었습니다 (기대값: 2 x 4 x 5 = 40).


In [3]:
# cue/results/table/comparison/Kd_reg_results.csv와 동일한 표기 방식(mean ± std 병합,
# 화살표로 방향 표시)을 사용한다.
METRIC_HEADERS = {'rmse': 'RMSE( ↓ )', 'r2': 'R2( ↑ )', 'pcc': 'PCC( ↑ )'}


def build_param_table(df_all, dataset, target, metrics=metrics_of_interest):
    """지정한 dataset/target에 대해 temperature 값별 metric의 fold 평균(mean)과 표준편차(std)를
    'mean ± std' 형태로 병합한 표를 만든다 (순위 없음).

    Returns:
        Optional[pd.DataFrame]: 컬럼이
            TAU | RMSE( ↓ ) | R2( ↑ ) | PCC( ↑ )
            순서인 표. TAU 오름차순으로 정렬한다.
            각 값은 '{mean:.3f} ± {std:.3f}' 문자열이다. 해당 dataset/target에 데이터가 하나도 없으면 None.
    """
    df_sub = df_all[(df_all['Dataset'] == dataset) & (df_all['Target'] == target)]
    if df_sub.empty:
        return None

    table = df_sub[['TAU']].copy()
    for m in metrics:
        if m in df_sub.columns and f'{m}_std' in df_sub.columns:
            header = METRIC_HEADERS.get(m, m.upper())
            table[header] = [f"{v:.3f} ± {s:.3f}" for v, s in zip(df_sub[m], df_sub[f'{m}_std'])]

    table = table.sort_values('TAU').reset_index(drop=True)

    return table


table_save_dir = os.path.join(cue_base, 'table', 'tau_comparison')
os.makedirs(table_save_dir, exist_ok=True)

# DIS -> dis(dissociation), INH -> inh(inhibition)
dataset_filenames = {'DIS': 'dis.csv', 'INH': 'inh.csv'}

param_tables = {}
for d_name in datasets:
    dataset_tables = []
    for target in targets:
        table = build_param_table(df_total, d_name, target)
        if table is None:
            continue
        param_tables[(d_name, target)] = table

        print(f"\n=== [{d_name} / {target}] ===")
        print(table.to_string(index=False))

        table_with_target = table.copy()
        table_with_target.insert(0, 'Target', target)
        dataset_tables.append(table_with_target)

    if not dataset_tables:
        continue

    dataset_table = pd.concat(dataset_tables, ignore_index=True)
    save_path = os.path.join(table_save_dir, dataset_filenames[d_name])
    dataset_table.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
    print(f"\n✅ 저장 완료: {save_path}")


=== [Kd / PI] ===
 TAU     RMSE( ↓ )       R2( ↑ )      PCC( ↑ )
0.25 0.243 ± 0.011 0.117 ± 0.058 0.361 ± 0.087
0.50 0.243 ± 0.011 0.114 ± 0.060 0.358 ± 0.088
1.00 0.243 ± 0.011 0.112 ± 0.062 0.355 ± 0.092
2.00 0.243 ± 0.011 0.111 ± 0.063 0.353 ± 0.095
4.00 0.244 ± 0.011 0.110 ± 0.063 0.352 ± 0.096

=== [Kd / Ssel] ===
 TAU     RMSE( ↓ )       R2( ↑ )      PCC( ↑ )
0.25 0.890 ± 0.048 0.074 ± 0.080 0.323 ± 0.088
0.50 0.894 ± 0.061 0.067 ± 0.094 0.318 ± 0.098
1.00 0.899 ± 0.070 0.058 ± 0.101 0.309 ± 0.103
2.00 0.902 ± 0.074 0.050 ± 0.119 0.306 ± 0.108
4.00 0.906 ± 0.073 0.040 ± 0.125 0.302 ± 0.108

=== [Kd / WS(alpha=2)] ===
 TAU     RMSE( ↓ )       R2( ↑ )      PCC( ↑ )
0.25 0.286 ± 0.014 0.195 ± 0.055 0.479 ± 0.044
0.50 0.284 ± 0.013 0.207 ± 0.076 0.489 ± 0.057
1.00 0.283 ± 0.012 0.210 ± 0.079 0.490 ± 0.059
2.00 0.283 ± 0.012 0.209 ± 0.080 0.490 ± 0.060
4.00 0.284 ± 0.013 0.206 ± 0.081 0.488 ± 0.060

=== [Kd / RS(k=2)] ===
 TAU     RMSE( ↓ )        R2( ↑ )      PCC( ↑ )
0.25 1.686 ± 0